In [1]:
import pandas as pd
from pathlib import Path
import math

# 1. Flexible Paths
BASE_DIR = Path.cwd().parent
if not (BASE_DIR / "data" / "processed" / "historical_matches.csv").exists():
    BASE_DIR = Path.cwd()

HISTORICAL_PATH = BASE_DIR / "data" / "processed" / "historical_matches.csv"
OUTPUT_MATCHES_PATH = BASE_DIR / "data" / "processed" / "historical_with_elo.csv"
OUTPUT_RATINGS_PATH = BASE_DIR / "data" / "processed" / "current_elo_ratings.csv"

# 2. Mathematical Elo Engine
class EloEngine:
    def __init__(self, k_factor=40, default_elo=1500):
        self.k = k_factor
        self.default = default_elo
        self.ratings = {}

    def get_rating(self, team):
        return self.ratings.get(team, self.default)

    def expected_result(self, rating_a, rating_b):
        return 1.0 / (1.0 + math.pow(10, (rating_b - rating_a) / 400.0))

    def update(self, home, away, home_score, away_score, tournament):
        rating_h = self.get_rating(home)
        rating_a = self.get_rating(away)

        # Determine actual result (1 for win, 0.5 for draw, 0 for loss)
        if home_score > away_score:
            actual_h, actual_a = 1.0, 0.0
        elif home_score < away_score:
            actual_h, actual_a = 0.0, 1.0
        else:
            actual_h, actual_a = 0.5, 0.5

        # Dynamic K-factor (World Cup matches cause bigger rating swings)
        k_multiplier = 1.5 if tournament == "FIFA World Cup" else 1.0
        current_k = self.k * k_multiplier

        exp_h = self.expected_result(rating_h, rating_a)
        exp_a = self.expected_result(rating_a, rating_h)

        # Apply rating updates
        self.ratings[home] = rating_h + current_k * (actual_h - exp_h)
        self.ratings[away] = rating_a + current_k * (actual_a - exp_a)

        # We return the ratings BEFORE the match started (crucial for ML training)
        return rating_h, rating_a

# 3. Execution Pipeline
print("[INFO] Loading clean historical data...")
df_all_history = pd.read_csv(HISTORICAL_PATH)
df_all_history["date"] = pd.to_datetime(df_all_history["date"])

# --------------------------------------------------------------------- #
# TIME FILTER - must run to completion BEFORE the Elo engine is even
# instantiated. Every team starts accumulating Elo from a *clean slate*
# (the engine's default rating) using only matches played from 2016
# onward, so the final ratings reflect the last 10 years of form -
# no 19th/20th-century results leak into the calculation.
# --------------------------------------------------------------------- #
ELO_CUTOFF_DATE = pd.Timestamp("2016-01-01")

matches_before_cutoff = (df_all_history["date"] < ELO_CUTOFF_DATE).sum()
df = df_all_history[df_all_history["date"] >= ELO_CUTOFF_DATE].copy()
df = df.sort_values(by="date").reset_index(drop=True)

print(f"[INFO] Dropped {matches_before_cutoff} matches played before {ELO_CUTOFF_DATE.date()}.")
print(f"[INFO] {len(df)} matches remain in scope for the Elo calculation.")

# Only NOW - with the dataset already fully restricted to 2016+ - do we
# create the engine and run the chronological update loop.
engine = EloEngine()
home_elos = []
away_elos = []

print(f"[INFO] Calculating Elo using matches from {ELO_CUTOFF_DATE.date()} onward (last 10 years of form)...")
# Loop through chronologically
for idx, row in df.iterrows():
    h_elo, a_elo = engine.update(
        row["home_team"], 
        row["away_team"], 
        row["home_score"], 
        row["away_score"],
        row["tournament"]
    )
    home_elos.append(h_elo)
    away_elos.append(a_elo)

# Append calculations to the dataset
df["home_elo_before"] = home_elos
df["away_elo_before"] = away_elos

# 4. Save Artifacts
df.to_csv(OUTPUT_MATCHES_PATH, index=False)

# --------------------------------------------------------------------- #
# ACTIVE-TEAMS FILTER - defensive safeguard against "zombie" entries
# (e.g. Yugoslavia, Czechoslovakia): explicitly drop any team from the
# final leaderboard that has not played a single match since
# 2016-01-01, even though the engine (fed only 2016+ fixtures above)
# should never have added such a team to begin with.
# --------------------------------------------------------------------- #
active_teams = set(df["home_team"]).union(set(df["away_team"]))

ratings_df = pd.DataFrame(list(engine.ratings.items()), columns=["team", "current_elo"])
inactive_teams = ratings_df.loc[~ratings_df["team"].isin(active_teams), "team"].tolist()
if inactive_teams:
    print(f"[INFO] Removing {len(inactive_teams)} inactive/defunct team(s) from the leaderboard: {inactive_teams}")
ratings_df = ratings_df[ratings_df["team"].isin(active_teams)].reset_index(drop=True)

ratings_df = ratings_df.sort_values(by="current_elo", ascending=False).reset_index(drop=True)
ratings_df.to_csv(OUTPUT_RATINGS_PATH, index=False)

print(f"[SUCCESS] Applied math to {len(df)} matches.")
print(f"[SUCCESS] {len(ratings_df)} active teams (played since {ELO_CUTOFF_DATE.date()}) in the final leaderboard.")
print("\n[LEADERBOARD] Top 10 International Teams globally right now:")
print(ratings_df.head(10).to_string(index=False))

[INFO] Loading clean historical data...
[INFO] Dropped 39453 matches played before 2016-01-01.
[INFO] 10031 matches remain in scope for the Elo calculation.
[INFO] Calculating Elo using matches from 2016-01-01 onward (last 10 years of form)...
[SUCCESS] Applied math to 10031 matches.
[SUCCESS] 294 active teams (played since 2016-01-01) in the final leaderboard.

[LEADERBOARD] Top 10 International Teams globally right now:
       team  current_elo
  Argentina  1976.325690
     France  1946.727957
      Spain  1945.197751
    Morocco  1910.335915
     Brazil  1899.010114
     Mexico  1896.896306
   Colombia  1862.584920
   Portugal  1858.590314
    England  1857.145877
Netherlands  1853.122038
